<a href="https://colab.research.google.com/github/Darwisher/Big-Data-Group_Activity/blob/main/Lab_Activity_3_Your_Data%3F_Our_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pyspark
from pyspark.sql import SparkSession

sprk = SparkSession.builder.appName('NBA_RDD_Project').getOrCreate()
sc = sprk.sparkContext

raw_rdd = sc.textFile("/content/2021-2022 NBA Player Stats - Regular.csv")
header = raw_rdd.first()

# Filter header and split by semicolon, as the CSV uses ';' as a delimiter
data_rdd = raw_rdd.filter(lambda line: line != header).map(lambda line: line.split(";"))

In [3]:

# Filter out 'TOT' (Total rows for traded players to avoid double counting)
# Team | Index 5: Games Played (G) | Index 29: Points Per Game (PTS)
clean_data = data_rdd.filter(lambda x: x[4] != 'TOT')

# Calculate Season Total for each player and map to (Team, PlayerSeasonTotal)
# Formula: (Games Played * Points Per Game)
team_season_rdd = clean_data.map(lambda x: (
    x[4],
    float(x[5]) * float(x[29])
)).partitionBy(30) # Hash Partitioning Strategy

In [4]:
# Step 3: ReduceByKey to sum all player season totals into a Team Season Total
total_season_scores = team_season_rdd.reduceByKey(lambda a, b: a + b) \
                                     .sortBy(lambda x: x[1], ascending=False)

print("--- Total Team Score for the Entire 2021-2022 Season ---")
for team, total in total_season_scores.collect():
    # Format with commas for better readability (e.g., 9,123.40)
    print(f"Team: {team:<5} | Total Season Points: {total:,.2f}")

--- Total Team Score for the Entire 2021-2022 Season ---
Team: MIN   | Total Season Points: 9,514.00
Team: MIL   | Total Season Points: 9,481.70
Team: MEM   | Total Season Points: 9,475.30
Team: CHO   | Total Season Points: 9,464.40
Team: PHO   | Total Season Points: 9,416.90
Team: ATL   | Total Season Points: 9,352.40
Team: UTA   | Total Season Points: 9,320.70
Team: SAS   | Total Season Points: 9,282.30
Team: BRK   | Total Season Points: 9,257.50
Team: DEN   | Total Season Points: 9,244.20
Team: LAL   | Total Season Points: 9,206.60
Team: BOS   | Total Season Points: 9,168.90
Team: CHI   | Total Season Points: 9,155.60
Team: IND   | Total Season Points: 9,145.80
Team: GSW   | Total Season Points: 9,110.20
Team: SAC   | Total Season Points: 9,049.90
Team: MIA   | Total Season Points: 9,024.50
Team: PHI   | Total Season Points: 9,015.50
Team: HOU   | Total Season Points: 9,001.00
Team: NOP   | Total Season Points: 8,973.80
Team: TOR   | Total Season Points: 8,967.10
Team: WAS   | Total

In [5]:
# PIPELINE 2: RANGE PARTITIONING (By Scoring Tier)
range_partitioned_rdd = data_rdd \
    .map(lambda x: (float(x[29]), x[1])) \
    .sortByKey() \
    .partitionBy(4)

In [6]:
# Transformation Pipeline:
# Filter for > 25 PPG and sort by Points in ASCENDING order
elite_players_asc = range_partitioned_rdd \
    .filter(lambda x: x[0] > 25.0) \
    .sortBy(lambda x: x[0], ascending=True) \
    .map(lambda x: f"Elite Scorer: {x[1]} ({x[0]} PPG)")

print("\n--- Pipeline 2: Elite Scorers (Range - Ascending) ---")
for player in elite_players_asc.collect():
    print(player)


--- Pipeline 2: Elite Scorers (Range - Ascending) ---
Elite Scorer: Stephen Curry (25.5 PPG)
Elite Scorer: Donovan Mitchell (25.9 PPG)
Elite Scorer: Devin Booker (26.8 PPG)
Elite Scorer: Jayson Tatum (26.9 PPG)
Elite Scorer: Nikola Joki? (27.1 PPG)
Elite Scorer: Kyrie Irving (27.4 PPG)
Elite Scorer: Ja Morant (27.4 PPG)
Elite Scorer: DeMar DeRozan (27.9 PPG)
Elite Scorer: Luka Don?i? (28.4 PPG)
Elite Scorer: Trae Young (28.4 PPG)
Elite Scorer: Giannis Antetokounmpo (29.9 PPG)
Elite Scorer: Kevin Durant (29.9 PPG)
Elite Scorer: LeBron James (30.3 PPG)
Elite Scorer: Joel Embiid (30.6 PPG)
